# Sakila DVD Rental: Combining Tables with pandas

**Author:** Maria Pupysheva · **Date:** October 2026

**Goal:** practise merging related tables in pandas and answer business questions that need
data from several tables. Results are cross-checked against earlier SQL queries.

**Tools:** Python 3.12 · pandas · SQLAlchemy · MySQL 8

## 1. Setup
Connect to the local MySQL server and load four tables into pandas DataFrames.
The password is requested at runtime, so it is never stored in the notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from getpass import getpass

pd.set_option("display.max_columns", 20)   # show all columns in wide tables

In [2]:
engine = create_engine(
    f"mysql+pymysql://root:{getpass('MySQL password: ')}@localhost:3306/sakila"
)

payments  = pd.read_sql("SELECT * FROM payment",  engine)
customers = pd.read_sql("SELECT * FROM customer", engine)
films     = pd.read_sql("SELECT * FROM film",     engine)
rentals   = pd.read_sql("SELECT * FROM rental",   engine)
inventory = pd.read_sql("SELECT * FROM inventory", engine)
film_category = pd.read_sql("SELECT * FROM film_category", engine)
category = pd.read_sql("SELECT * FROM category", engine)

# Size of each table
pd.DataFrame(
    {
        "rows":    [len(df) for df in (payments, customers, films, rentals, inventory, film_category, category)],
        "columns": [df.shape[1] for df in (payments, customers, films, rentals, inventory, film_category, category)],
    },
    index=["payments", "customers", "films", "rentals", "inventory", "film_category", "category"],
)

,rows,columns
payments,16044,7
customers,599,9
films,1000,13
rentals,16044,7
inventory,4581,4
film_category,1000,3
category,16,3


## 2. Table relationships

### Q1. How the tables connect
Before merging, I identified the key column that links each pair of tables.
Merging on the wrong column silently creates wrong or duplicated rows, so this check
comes first. For each pair, I listed the column names the two tables share
(set intersection), then judged which shared column is a real key.

In [3]:
# Columns shared by each pair of tables
pairs = {
    "payments ↔ rentals":       (payments, rentals),
    "rentals ↔ inventory":      (rentals, inventory),
    "inventory ↔ film_category": (inventory, film_category),
    "film_category ↔ category": (film_category, category),
}

for name, (left, right) in pairs.items():
    print(f"{name:28} {sorted(set(left.columns) & set(right.columns))}")

payments ↔ rentals           ['customer_id', 'last_update', 'rental_id', 'staff_id']
rentals ↔ inventory          ['inventory_id', 'last_update']
inventory ↔ film_category    ['film_id', 'last_update']
film_category ↔ category     ['category_id', 'last_update']


**Conclusion:** the keys for the merge chain are

| Tables | Key | Relationship |
|---|---|---|
| payments ↔ rentals | `rental_id` | each payment belongs to one rental |
| rentals ↔ inventory | `inventory_id` | each rental is of one physical DVD copy |
| inventory ↔ film_category | `film_id` | each copy is of one film, and each film has one category |
| film_category ↔ category | `category_id` | each film category points to one category name |

**Why not the other shared columns:**
- `last_update` appears in every table, but it is only a timestamp of when the row was
  last changed. Equal timestamps mean nothing, so it is not a key.
- `customer_id` and `staff_id` are shared by payments and rentals, but they are not
  unique per row: one customer has many payments and many rentals. Merging on them
  alone would match every payment with every rental of that customer and multiply
  the rows. `rental_id` links each payment to exactly one rental.

When merging, `last_update` columns would clash, so pandas would add suffixes
(`_x`, `_y`); selecting only the needed columns before merging avoids this.

In [4]:
# Revenue per film category: payment → rental → inventory → film_category → category
revenue_by_category = (
    payments[["rental_id", "amount"]]
    .merge(rentals[["rental_id", "inventory_id"]], on="rental_id")
    .merge(inventory[["inventory_id", "film_id"]], on="inventory_id")
    .merge(film_category[["film_id", "category_id"]], on="film_id")
    .merge(category[["category_id", "name"]], on="category_id")
    .rename(columns={"name": "category"})
    .groupby("category")
    .agg(rentals=("rental_id", "count"), revenue=("amount", "sum"))
    .sort_values("revenue", ascending=False)
    .round(2)
    .reset_index()
)

revenue_by_category["rentals_rank"] = revenue_by_category["rentals"].rank(ascending=False).astype(int)
revenue_by_category["avg_per_rental"] = (revenue_by_category["revenue"] / revenue_by_category["rentals"]).round(2)

# Sanity check: revenue covered by the merge vs total revenue
print(f"Revenue in merge: {revenue_by_category['revenue'].sum():,.2f}")
print(f"Total revenue:    {payments['amount'].sum():,.2f}")

revenue_by_category

Revenue in merge: 67,406.56
Total revenue:    67,406.56


,category,rentals,revenue,rentals_rank,avg_per_rental
0,Sports,1179,5314.21,1,4.51
1,Sci-Fi,1101,4756.98,4,4.32
2,Animation,1166,4656.30,2,3.99
3,Drama,1060,4587.39,6,4.33
4,Comedy,941,4383.58,11,4.66
5,Action,1112,4375.85,3,3.94
6,New,940,4351.62,12,4.63
7,Games,969,4281.33,9,4.42
8,Foreign,1033,4270.67,8,4.13
9,Family,1096,4226.07,5,3.86


**Observation:** Revenue is fairly evenly spread across the 16 categories (≈5–8% each).
Sports earns the most (5,314, ≈7.9%), about 55% more than Music (3,418, ≈5.1%).
Rental volume and revenue do not rank identically, because the average amount per rental
differs: Comedy (≈4.66) and New (≈4.63) rank low by rentals but high by revenue, while
Family (≈3.86), Children and Classics (≈3.87–3.88) earn the least per rental. Action is
3rd by rentals but only 6th by revenue.

**Data check:** the merged revenue equals total revenue (67,406.56), so every payment was
linked to exactly one category; no rows were lost or duplicated.

In [5]:
revenue_by_rating = (
    films[["film_id", "rating"]]
    .merge(inventory[["film_id", "inventory_id"]], on="film_id")
    .merge(rentals[["inventory_id", "rental_id"]], on="inventory_id")
    .merge(payments[["rental_id", "amount"]], on="rental_id")
    .groupby("rating")
    .agg(rentals=("rental_id", "count"), revenue=("amount", "sum"))
    .sort_values("revenue", ascending=False)
    .round(2)
    .reset_index()
)

revenue_by_rating["avg_per_rental"] = (revenue_by_rating["revenue"] / revenue_by_rating["rentals"]).round(2)

# Sanity check: revenue covered by the merge vs total revenue
print(f"Revenue in merge: {revenue_by_rating['revenue'].sum():,.2f}")
print(f"Total revenue:    {payments['amount'].sum():,.2f}")

revenue_by_rating

Revenue in merge: 67,406.56
Total revenue:    67,406.56


,rating,rentals,revenue,avg_per_rental
0,PG-13,3585,15259.16,4.26
1,NC-17,3293,13875.07,4.21
2,PG,3212,13337.91,4.15
3,R,3181,13270.19,4.17
4,G,2773,11664.23,4.21


In [6]:
revenue_by_rating["revenue_share"] = revenue_by_rating["revenue"] / revenue_by_rating["revenue"].sum()
revenue_by_rating

,rating,rentals,revenue,avg_per_rental,revenue_share
0,PG-13,3585,15259.16,4.26,0.226375
1,NC-17,3293,13875.07,4.21,0.205842
2,PG,3212,13337.91,4.15,0.197873
3,R,3181,13270.19,4.17,0.196868
4,G,2773,11664.23,4.21,0.173043


**Observation:** Revenue by rating closely follows the size of the catalogue: each rating's
share of revenue (17.3–22.6%) is within about half a percentage point of its share of films
(Q12 in notebook 1). PG-13 earns the most (15,259) because it has the most films, and G the
least (11,664) because it has the fewest. The average amount per rental is almost identical
across ratings (4.15–4.26), so rating does not affect price or demand per film.

**Data check:** the merged revenue equals total revenue (67,406.56).

In [7]:
# Version 1: store where the customer is registered
revenue_by_customer_store = (
    payments[["payment_id","customer_id", "amount"]]
    .merge(customers[["customer_id", "store_id"]], on="customer_id")
    .groupby("store_id")
    .agg(rentals=("payment_id", "count"), revenue=("amount", "sum"))
    .sort_values("revenue", ascending=False)
    .round(2)
    .reset_index()
)

revenue_by_customer_store

,store_id,rentals,revenue
0,1,8747,36997.53
1,2,7297,30409.03


In [8]:
# Version 2: store that owns the rented DVD copy
revenue_by_copy_store = (
    inventory[["inventory_id", "store_id"]]
    .merge(rentals[["inventory_id", "rental_id"]], on="inventory_id")
    .merge(payments[["rental_id", "amount"]], on="rental_id")
    .groupby("store_id")
    .agg(rentals=("rental_id", "count"), revenue=("amount", "sum"))
    .round(2)
    .reset_index()
)

revenue_by_copy_store

,store_id,rentals,revenue
0,1,7923,33679.79
1,2,8121,33726.77


In [9]:
# Payments by customer's home store (rows) × store that owns the copy (columns)
store_check = (
    payments[["payment_id", "customer_id", "rental_id", "amount"]]
    .merge(customers[["customer_id", "store_id"]].rename(columns={"store_id": "customer_store"}), on="customer_id")
    .merge(rentals[["rental_id", "inventory_id"]], on="rental_id")
    .merge(inventory[["inventory_id", "store_id"]].rename(columns={"store_id": "copy_store"}), on="inventory_id")
)

pd.crosstab(store_check["customer_store"], store_check["copy_store"], margins=True)

copy_store,1,2,All
customer_store,,,
1,4326,4421,8747
2,3597,3700,7297
All,7923,8121,16044


In [10]:
customers["store_id"].value_counts()

store_id
1    326
2    273
Name: count, dtype: int64

**Observation:** The answer depends on how "store" is defined.
- By the customer's home store, store 1 earns more (36,998 vs 30,409, ≈55% of revenue),
  because more customers are registered there.
- By the store that owns the rented copy, the two stores earn almost exactly the same
  (33,680 vs 33,727).
- The cross-tab shows why: customers rent from both stores about equally, whatever their
  home store (store 1 customers rent 51% of their DVDs from store 2; store 2 customers
  rent 49% from store 1).

For stock and shelf decisions, revenue by copy store is the right measure, and the two stores
perform the same. 

In [11]:
rentals_per_customer = rentals.groupby("customer_id").agg(rentals=("rental_id", "count"))
paid_per_customer    = payments.groupby("customer_id").agg(total_paid=("amount", "sum"))

top_10_renters = (
    rentals_per_customer
    .merge(paid_per_customer, on="customer_id")
    .nlargest(10, "rentals", keep="all")
    .reset_index()
    .merge(customers[["customer_id", "first_name", "last_name"]], on="customer_id")
)

top_10_renters

,customer_id,rentals,total_paid,first_name,last_name
0,148,46,216.54,ELEANOR,HUNT
1,526,45,221.55,KARL,SEAL
2,144,42,195.58,CLARA,SHAW
3,236,42,175.58,MARCIA,DEAN
4,75,41,155.59,TAMMY,SANDERS
5,197,40,154.60,SUE,PETERS
6,469,40,177.60,WESLEY,BULL
7,137,39,194.61,RHONDA,KENNEDY
8,178,39,194.61,MARION,SNYDER
9,468,39,175.61,TIM,CARY


In [12]:
rentals_per_customer.merge(paid_per_customer, on="customer_id").corr().round(2)

,rentals,total_paid
rentals,1.00,0.87
total_paid,0.87,1.00


**Observation:** The biggest renters are mostly also the biggest spenders: 8 of the top 10
customers by rentals are in the top 10 by total paid (notebook 1, Q13). Rental count is the
main driver of spending. The exceptions are TAMMY SANDERS and SUE PETERS, who rented 40–41
times but paid only ~155 each (≈3.8 per rental vs ≈4.7 for KARL SEAL), so they mostly rent
cheaper titles.

In [13]:
films_no_copies = (
    films[["film_id", "title"]]
    .merge(inventory[["film_id", "inventory_id"]], on="film_id", how="left", indicator=True)
    .query('_merge == "left_only"')
    [["film_id", "title"]]
    .sort_values("title")
)

print(f"Films with no copies in inventory: {len(films_no_copies)}")
films_no_copies

Films with no copies in inventory: 42


,film_id,title
70,14,ALICE FANTASIA
156,33,APOLLO TEEN
168,36,ARGONAUTS TOWN
176,38,ARK RIDGEMONT
188,41,ARSENIC INDEPENDENCE
399,87,BOONDOCK BALLROOM
490,108,BUTCH PANTHER
594,128,CATCH AMISTAD
673,144,CHINATOWN GLADIATOR
686,148,CHOCOLATE DUCK


In [14]:
films_no_rentals = (
    films[["film_id", "title"]]
    .merge(inventory[["film_id", "inventory_id"]], on="film_id")
    .merge(rentals[["inventory_id", "rental_id"]], on="inventory_id", how="left")
    .groupby(["film_id", "title"])
    .agg(rentals=("rental_id", "count"))
    .query('rentals == 0')
)

films_no_rentals

,,rentals
film_id,title,


**Observation:** 42 of the 1,000 films were never rented, and all of them are films with no
copies in inventory (matching the SQL result). Every film the store actually stocks was rented
at least once. So unrented titles are a stock issue, not a demand issue: these 42 films cannot
be rented because the store does not own a copy.

## Summary

**Key findings**
- **Categories:** Sports leads both revenue (5,314, ≈7.9%) and rentals (1,179), but revenue is
  fairly evenly spread across the 16 categories (≈5–8% each). Average price per rental differs:
  Comedy and New earn ≈4.6 per rental, Family, Children and Classics ≈3.9 (Q2).
- **Ratings:** revenue by rating follows catalogue size. PG-13 earns the most (15,259) because it
  has the most films (223); revenue per film and price per rental are almost the same for every
  rating (Q3).
- **Stores:** the answer depends on the definition. By home store, store 1 earns more
  (≈55%); by the store that owns the copy, both earn the same (≈50% each). Customers do not
  stick to their home store: about half of their rentals come from the other store (Q4).
- **Customers:** renting more strongly goes with spending more (correlation 0.82 across all
  599 customers); 8 of the top 10 renters are also top-10 spenders. Exceptions such as
  TAMMY SANDERS and SUE PETERS rent often but choose cheaper titles (Q5).
- **Unrented films:** all 42 never-rented films have no copies in inventory, and every stocked
  film was rented at least once, so unrented titles are a stock issue, not a demand issue (Q6).

**Data checks**
- In every merge, the merged revenue matched total revenue (67,406.56), so no payments were
  lost or duplicated.
- Results match the earlier SQL queries (category revenue, 42 films without copies).